# EFUS 2017 — Raw temperature & sample descriptive statistics

Descriptive summary of the raw indoor/outdoor temperature records and the
composition of the dwelling sample, reported for the **National** dataset
(all Government Office Regions pooled) and the **London** subset (`gorEHS_efus == 7`),
for both the **living-room** (`LR`) and **bedroom** (`BED1`–`BED3`) sensor extracts.

For each of indoor (`T_in`) and outdoor (`T_out`) temperature the tables give:
number of measurements, mean, min, max, range (max − min), standard deviation,
and the 90 % confidence interval for the mean.

> **Caveat on the 90 % CI.** The half-hourly records are strongly autocorrelated
> within a dwelling, so the naïve standard error `s/√n` understates the true
> sampling uncertainty and the CI below is correspondingly *too narrow*. It is
> reported as a conventional descriptive figure, not an inferential claim.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

ROOT = "."
BLDG_CSV = f"{ROOT}/test_data/ukda_9434_csv_r/csv/selected_interview_responses_caseid.csv"

# Building characteristics, one row per dwelling (CaseID)
BLDG_COLS = [
    "CaseID", "gorEHS_efus", "dwtype_efus", "dwage_efus", "WallType2x_efus",
    "InsulatedWalls_efus", "FullyDblGlz_efus", "floor6x_efus",
    "EPceeb12e_efus", "AnyCooling",
]
bldg = pd.read_csv(BLDG_CSV, usecols=BLDG_COLS).drop_duplicates("CaseID")
n_efus    = bldg["CaseID"].nunique()
n_efus_ldn = (bldg["gorEHS_efus"] == 7).sum()
print(f"EFUS 2017 interview sample : {n_efus:,} dwellings "
      f"({n_efus_ldn} in London, GOR 7)")

In [ ]:
def load_room(path):
    """Load a room parquet, drop out-of-range temps, merge building chars."""
    df = pd.read_parquet(path)
    df = df[df["T_in"].between(-10, 40) & df["T_out"].between(-10, 40)].copy()
    df = df.merge(bldg, on="CaseID", how="left")
    return df

rooms = {
    "Living room": load_room(f"{ROOT}/efus_indoor_outdoor_livingroom.parquet"),
    "Bedroom":     load_room(f"{ROOT}/efus_indoor_outdoor_bedroom.parquet"),
}

def region_subset(df, region):
    return df if region == "National" else df[df["gorEHS_efus"] == 7]

REGIONS = ["National", "London"]

## 0. Sample overview

Number of monitored **buildings** (dwellings), total temperature measurements, the
spread of measurements per building, and the monitoring window — by room and region.
Note the two rooms are different sensor sub-samples of the EFUS interview cohort
(not all dwellings carry both a living-room and a bedroom sensor).

In [ ]:
def overview(df, reg):
    sub = region_subset(df, reg)
    npd = sub.groupby("CaseID").size()
    return {
        "Buildings (dwellings)":        f"{sub['CaseID'].nunique():,}",
        "% of EFUS sample":             f"{sub['CaseID'].nunique()/ (n_efus if reg=='National' else n_efus_ldn)*100:.0f}%",
        "Total measurements":           f"{len(sub):,}",
        "Measurements / building (mean)":   f"{npd.mean():,.0f}",
        "Measurements / building (median)": f"{npd.median():,.0f}",
        "Measurements / building (min–max)":f"{npd.min():,}–{npd.max():,}",
        "Monitoring period":            f"{sub['hour'].min():%Y-%m-%d} – {sub['hour'].max():%Y-%m-%d}",
    }

ov = {(rname, reg): overview(df, reg)
      for rname, df in rooms.items() for reg in REGIONS}
ov_tab = pd.DataFrame(ov)
ov_tab.columns = pd.MultiIndex.from_tuples(ov_tab.columns)
ov_tab

## 1. Temperature descriptive statistics

In [ ]:
def temp_stats(series, ci=0.90):
    """Descriptive stats for one temperature series, returned pre-formatted."""
    s = series.dropna().to_numpy()
    n = s.size
    mean, sd = s.mean(), s.std(ddof=1)
    se = sd / np.sqrt(n)
    z  = stats.norm.ppf(0.5 + ci / 2)          # 1.645 for 90%
    lo, hi = mean - z * se, mean + z * se
    return {
        "N measurements":      f"{n:,}",
        "Mean (°C)":           f"{mean:.2f}",
        "Min (°C)":            f"{s.min():.2f}",
        "Max (°C)":            f"{s.max():.2f}",
        "Range (°C)":          f"{s.max() - s.min():.2f}",
        "Std (°C)":            f"{sd:.2f}",
        "90% CI of mean (°C)": f"{lo:.3f} – {hi:.3f}",
    }

def temp_table(var):
    cols = {(rname, reg): temp_stats(region_subset(df, reg)[var])
            for rname, df in rooms.items() for reg in REGIONS}
    tab = pd.DataFrame(cols)
    tab.columns = pd.MultiIndex.from_tuples(tab.columns)
    return tab

print("Indoor temperature  (T_in)")
tin_tab = temp_table("T_in")
tin_tab

In [ ]:
print("Outdoor temperature  (T_out)  — nearest MIDAS Open station")
tout_tab = temp_table("T_out")
tout_tab

### Per-dwelling temperature summary (90 % CI across dwellings)

Each **building** is first reduced to its own maximum, minimum, and mean indoor
temperature (`T_in`); the table then reports the **average of those per-building
values across dwellings**, with a 90 % confidence interval. Because each dwelling
contributes a single value, this CI reflects genuine between-building variation
(unlike the per-measurement CI above, which is inflated by autocorrelation).
Cells are `mean (low – high)` in °C; *n* = number of dwellings.

In [ ]:
def per_dwelling_ci(df, reg, var="T_in", ci=0.90):
    sub = region_subset(df, reg)
    g = sub.groupby("CaseID")[var]
    per = pd.DataFrame({"Max temp (°C)": g.max(),
                        "Min temp (°C)": g.min(),
                        "Avg temp (°C)": g.mean()})
    z = stats.norm.ppf(0.5 + ci / 2)
    out = {}
    for label in per.columns:
        x = per[label].to_numpy()
        m, se = x.mean(), x.std(ddof=1) / np.sqrt(len(x))
        out[label] = f"{m:.2f} ({m - z*se:.2f} – {m + z*se:.2f})"
    out["n (dwellings)"] = f"{len(per):,}"
    return out

cols = {(rname, reg): per_dwelling_ci(df, reg)
        for rname, df in rooms.items() for reg in REGIONS}
dwell_tab = pd.DataFrame(cols)
dwell_tab.columns = pd.MultiIndex.from_tuples(dwell_tab.columns)
print("Indoor temperature (T_in) — per-dwelling max / min / avg, 90% CI across dwellings")
dwell_tab

## 2. Composition of dwelling characteristics

One row per **dwelling** (unique `CaseID`) present in each room's extract, so the
counts reflect the dwellings actually contributing temperature records. Each cell
is `count (percent of column total)`. A *Missing/unknown* row is shown where a
characteristic is not recorded for some dwellings, so every column sums to 100 %.

In [ ]:
# Coding -> human labels (no numbers), matching the report table
LABELS = {
    "dwtype_efus":      {1:"Detached",2:"Semi-detached",3:"End-terrace",
                         4:"Mid-terrace",5:"Bungalow",6:"Flat"},
    "dwage_efus":       {1:"Pre-1919",2:"1919–44",3:"1944–64",4:"1964–80",
                         5:"1980–90",6:"1990–2002",7:"Post-2002"},
    "WallType2x_efus":  {1:"Solid wall",2:"Cavity wall"},
    "InsulatedWalls_efus":{0:"Walls not insulated",1:"Walls insulated"},
    "FullyDblGlz_efus": {0:"Not fully dbl-glazed",1:"Fully double-glazed"},
    "EPceeb12e_efus":   {1:"EPC C or better",2:"EPC D",3:"EPC E",4:"EPC F/G"},
    "AnyCooling":       {0:"No cooling",1:"Any cooling"},
}
CHAR_TITLES = {
    "dwtype_efus":"Dwelling type", "dwage_efus":"Dwelling age",
    "WallType2x_efus":"Wall construction", "InsulatedWalls_efus":"Wall insulation",
    "FullyDblGlz_efus":"Glazing", "EPceeb12e_efus":"EPC band",
    "AnyCooling":"Cooling",
}

def dwelling_frame(df, reg):
    return region_subset(df, reg).drop_duplicates("CaseID")

def composition(char):
    order = list(LABELS[char].values())
    cols, any_missing = {}, False
    for rname, df in rooms.items():
        for reg in REGIONS:
            d = dwelling_frame(df, reg)
            tot = len(d)
            vc = d[char].value_counts(dropna=False)
            labelled, cells = 0, {}
            for code, lab in LABELS[char].items():
                c = int(vc.get(code, 0))
                labelled += c
                cells[lab] = f"{c} ({c/tot*100:.1f}%)"
            missing = tot - labelled
            any_missing = any_missing or missing > 0
            cells["Missing/unknown"] = f"{missing} ({missing/tot*100:.1f}%)"
            cells["Total dwellings"] = f"{tot}"
            cols[(rname, reg)] = cells
    rows = order + (["Missing/unknown"] if any_missing else []) + ["Total dwellings"]
    tab = pd.DataFrame(cols).reindex(rows)
    tab.columns = pd.MultiIndex.from_tuples(tab.columns)
    tab.index.name = CHAR_TITLES[char]
    return tab

for char in ["dwtype_efus","dwage_efus","WallType2x_efus","InsulatedWalls_efus",
             "FullyDblGlz_efus","EPceeb12e_efus","AnyCooling"]:
    print(f"\n=== {CHAR_TITLES[char]} ===")
    display(composition(char))

## 3. Number of houses per building characteristic

Single consolidated table: count of **dwellings** in each category, by room and
region, for the six fabric/built-form characteristics. Rows are grouped by
characteristic; the final row gives the total dwellings per column. (These six
characteristics have no missing values, so each block sums to the total.)

In [ ]:
COUNT_CHARS = ["dwtype_efus", "dwage_efus", "WallType2x_efus",
               "InsulatedWalls_efus", "FullyDblGlz_efus", "EPceeb12e_efus"]

def counts_table(chars):
    blocks = []
    for char in chars:
        order = list(LABELS[char].values())
        cols = {}
        for rname, df in rooms.items():
            for reg in REGIONS:
                vc = dwelling_frame(df, reg)[char].value_counts(dropna=False)
                cols[(rname, reg)] = {lab: int(vc.get(code, 0))
                                      for code, lab in LABELS[char].items()}
        blk = pd.DataFrame(cols).reindex(order)
        blk.index = pd.MultiIndex.from_product([[CHAR_TITLES[char]], order])
        blocks.append(blk)
    tot = pd.DataFrame({(rname, reg): {"Dwellings": len(dwelling_frame(df, reg))}
                        for rname, df in rooms.items() for reg in REGIONS})
    tot.index = pd.MultiIndex.from_product([["Total"], ["Dwellings"]])
    tab = pd.concat(blocks + [tot])
    tab.columns = pd.MultiIndex.from_tuples(tab.columns)
    tab.index.names = ["Characteristic", "Category"]
    return tab

house_counts = counts_table(COUNT_CHARS)
house_counts

## 4. LaTeX export (optional)

Uncomment to dump any table to LaTeX for the report (e.g. the sample overview or the
indoor-temperature summary). Requires `booktabs` in the document preamble.

In [ ]:
# print(house_counts.to_latex(multicolumn=True, multicolumn_format="c"))
# print(dwell_tab.to_latex(multicolumn=True, multicolumn_format="c"))
# print(ov_tab.to_latex(multicolumn=True, multicolumn_format="c"))
# print(tin_tab.to_latex(multicolumn=True, multicolumn_format="c"))
# print(composition("dwtype_efus").to_latex())